# NER Pipeline — HuggingFace Transformers + GPU

Model: `Jean-Baptiste/roberta-large-ner-english`  
Good for noisy/historical newspaper text. Handles PERSON, ORG, LOC/GPE.

Swap to `dslim/bert-large-NER` if you want a lighter model with slightly faster throughput.

In [1]:
import os
import json
import logging
import torch
import pandas as pd

from collections import Counter
from pathlib import Path
from tqdm import tqdm
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification

C:\Users\johna\miniforge3\envs\textmining2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Device setup ──────────────────────────────────────────────────────────────
# Priority: CUDA (Nvidia) > XPU (Intel Arc/Data Center) > CPU
if torch.cuda.is_available():
    DEVICE = 0          # HuggingFace pipeline uses integer device index for CUDA
    DEVICE_NAME = torch.cuda.get_device_name(0)
elif hasattr(torch, 'xpu') and torch.xpu.is_available():
    # Intel XPU — pipeline doesn't accept 'xpu' string directly,
    # so we move the model manually after loading (see model cell below).
    DEVICE = 'xpu'
    DEVICE_NAME = 'Intel XPU'
else:
    DEVICE = -1         # CPU fallback
    DEVICE_NAME = 'CPU'

print(f'PyTorch version : {torch.__version__}')
print(f'Selected device : {DEVICE_NAME}')

PyTorch version : 2.7.0+xpu
Selected device : Intel XPU


In [3]:
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

In [4]:
# ── Configuration ─────────────────────────────────────────────────────────────
TARGET_DIR  = 'E:/Callproject/extracted_json'

# Use a local path if you have the weights cached, otherwise the HuggingFace
# Hub name below will download on first run.
MODEL_NAME  = 'Jean-Baptiste/roberta-large-ner-english'

# How many texts to feed the model at once.
# Increase if you have >8 GB VRAM; decrease if you hit OOM errors.
BATCH_SIZE  = 32

# Aggregation strategy: 'simple' merges sub-word tokens back into words.
# 'first' and 'max' are alternatives if you see odd splits.
AGGREGATION = 'simple'

In [5]:
# ── Model loading ─────────────────────────────────────────────────────────────
logging.info(f'Loading model: {MODEL_NAME}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, add_prefix_space=True)
model     = AutoModelForTokenClassification.from_pretrained(MODEL_NAME)
model.eval()

if DEVICE == 'xpu':
    # Intel extension must be imported before moving the model
    import intel_extension_for_pytorch as ipex
    model = model.to('xpu')
    # Wrap with IPEX for optimised inference
    model = ipex.optimize(model, dtype=torch.float32)
    # HuggingFace pipeline needs a plain device string when passing a pre-loaded model
    pipeline_device = 'xpu'
    print(f'Intel PyTorch Extension version: {ipex.__version__}')
else:
    pipeline_device = DEVICE   # int (0 for CUDA) or -1 (CPU)

ner_pipe = pipeline(
    'ner',
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy=AGGREGATION,
    device=pipeline_device,
)

logging.info('Pipeline ready.')

INFO: Loading model: Jean-Baptiste/roberta-large-ner-english
C:\Users\johna\miniforge3\envs\textmining2\Lib\site-packages\intel_extension_for_pytorch\nn\utils\_weight_prepack.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-04-08 16:45:25,041 - _logger.py - IPEX - INFO - Currently split master weight for xpu only support sgd
2026-04-08 16:45:25,047 - _logger.py - IPEX - INFO - Conv BatchNorm folding failed during the optimize process.
2026-04-08 16:45:25,051 - _logger.py - IPEX - INFO - Linear BatchNorm folding failed during the optimize process.
Device set to use xpu
INFO: Pipeline ready.


Intel PyTorch Extension version: 2.7.10+xpu


In [6]:
# ── Label mapping ─────────────────────────────────────────────────────────────
# roberta-large-ner uses B-/I- prefixed labels.
# Map them to the three categories we care about.
LABEL_MAP = {
    'PER':  'PERSON',
    'ORG':  'ORG',
    'LOC':  'GPE',
    'MISC': None,      # drop miscellaneous
}

def map_label(raw_label: str):
    """Strips B-/I- prefix and maps to our category names."""
    tag = raw_label.split('-')[-1]   # 'B-PER' → 'PER', 'PER' → 'PER'
    return LABEL_MAP.get(tag)


# ── File helpers ──────────────────────────────────────────────────────────────
def get_file_list(root_path, limit=None):
    """Collect JSON paths; apply optional cap for testing."""
    print('Scanning directory for files...')
    files = list(Path(root_path).rglob('*.json'))
    if limit:
        print(f'Test mode: limiting to first {limit} files.')
        return files[:limit]
    return files


def extract_text(file_path) -> str:
    """Pull all paragraph text from one JSON file."""
    try:
        with open(file_path, 'r', encoding='utf-8') as fh:
            data = json.load(fh)
        parts = [
            para['text']
            for page  in data.get('pages',  [])
            for frame in page.get('frames', [])
            for para  in frame.get('paragraphs', [])
            if para.get('text')
        ]
        return ' '.join(parts)
    except Exception:
        return ''


def chunk_text(text: str, max_chars: int = 450) -> list[str]:
    """
    Split long text into word-boundary chunks that fit within the model's
    512-token limit. 450 chars is a safe heuristic for ~380 tokens.
    """
    words  = text.split()
    chunks, current = [], []
    length = 0
    for word in words:
        if length + len(word) + 1 > max_chars and current:
            chunks.append(' '.join(current))
            current, length = [], 0
        current.append(word)
        length += len(word) + 1
    if current:
        chunks.append(' '.join(current))
    return chunks if chunks else ['']

In [26]:
# ── Main processing loop ──────────────────────────────────────────────────────
def process_dataset(root_path, file_limit=None):
    files = get_file_list(root_path, limit=file_limit)
    counts = {'PERSON': Counter(), 'GPE': Counter(), 'ORG': Counter()}

    print(f'Running NER on {len(files):,} files...')

    # Build a flat list of (chunk, file_index) so we can batch across files.
    # This is more GPU-efficient than processing file-by-file.
    all_chunks = []
    for fp in files:
        text = extract_text(fp)
        all_chunks.extend(chunk_text(text))
    print(type(all_chunks))
    print(f'  → {len(all_chunks):,} text chunks to process')
    all_chunks = [c for c in all_chunks if c and c.strip()]

    # Verify no empty strings slipped through
    assert all(c.strip() for c in all_chunks), "Empty chunk found after filtering!"
    print(f"Clean chunks: {len(all_chunks)}")    

    # Stream chunks through the pipeline in batches.
    # tqdm wraps the generator for a live progress bar.
    if not all_chunks:
        raise ValueError(
            f"No text chunks were produced from '{TARGET_DIR}'. "
            "Check that the directory contains readable files and that "
            "your chunking logic is working correctly."
        )

    print(f"Chunks count: {len(all_chunks)}")
    print(f"First chunk type: {type(all_chunks[0])}")
    print(f"First chunk repr: {repr(all_chunks[0][:100])}")
    
    # Test the pipeline on a single string first, bypassing iter()
    test_result = ner_pipe("This is a test sentence about London and Microsoft.")
    print(f"Pipeline smoke test: {test_result}")

    for batch_result in tqdm(
        ner_pipe(all_chunks, batch_size=BATCH_SIZE),
        total=len(all_chunks),
        desc='NER progress',
        unit='chunk',
    ):


    # for batch_result in tqdm(
    #     ner_pipe(
    #         iter(all_chunks),
    #         batch_size=BATCH_SIZE,
    #     ),
    #     total=len(all_chunks),
    #     desc='NER progress',
    #     unit='chunk',
    # ):
        for entity in batch_result:
            category = map_label(entity['entity_group'])
            if category:
                name = entity['word'].strip()
                # Skip very short fragments (single letters, punctuation)
                if len(name) > 1:
                    counts[category][name] += 1

    return counts

In [27]:
# ── Run ───────────────────────────────────────────────────────────────────────
# Set file_limit=None to process all ~100k files.

results = process_dataset(TARGET_DIR, file_limit=100)

# ── Export ────────────────────────────────────────────────────────────────────
all_data = [
    {'Category': cat, 'Entity': name, 'Frequency': freq}
    for cat, counter in results.items()
    for name, freq in counter.most_common()
]

df_results = pd.DataFrame(all_data)
df_results.to_csv('ner_results_summary.csv', index=False)
print("Results exported to 'ner_results_summary.csv'")
print(df_results.sort_values('Frequency', ascending=False).head(15).to_string(index=False))

Scanning directory for files...
Test mode: limiting to first 100 files.
Running NER on 100 files...
<class 'list'>
  → 480 text chunks to process
Clean chunks: 479
Chunks count: 479
First chunk type: <class 'str'>
First chunk repr: 'Because sometimes there are no elevators. Are you missing out on life because of chronic knee, hip o'
Pipeline smoke test: [{'entity_group': 'LOC', 'score': np.float32(0.9962276), 'word': ' London', 'start': 30, 'end': 36}, {'entity_group': 'ORG', 'score': np.float32(0.9969189), 'word': ' Microsoft', 'start': 41, 'end': 50}]


NER progress: 100%|███████████████████████████████████████████████████████████| 479/479 [00:00<00:00, 169170.73chunk/s]

Results exported to 'ner_results_summary.csv'
Category                    Entity  Frequency
     GPE               Kansas City        181
     GPE                        Mo         92
     GPE                       Kas         59
     GPE                  Missouri         33
  PERSON                Cunningham         29
     ORG                    Chiefs         23
     GPE                    Kansas         22
     GPE                    Harlem         21
     GPE                       Mo.         19
     GPE Providence Medical Center         15
  PERSON                        D.         15
  PERSON                      Rice         15
  PERSON                    Herron         15
  PERSON        George Robinson, M         14
     ORG         providence-health         14


In [ ]:
# ── Quick spot-check on a single text ─────────────────────────────────────────
sample = 'President Truman met with General MacArthur in Kansas City, Missouri, on behalf of the United Nations.'
for ent in ner_pipe(sample):
    print(f"{ent['entity_group']:6s}  {ent['word']:30s}  score={ent['score']:.3f}")